# SARIMAX: petrol use and roadside NO₂

This notebook follows `model_ready/README.md`. It tunes only on the pre-validation training period, evaluates the held-out validation year, refits on training plus validation data, and forecasts the policy period. It runs both the main and robustness (extra) specifications.

Install the dependencies once if needed: `python -m pip install pandas numpy matplotlib statsmodels`. Run all cells from the repository root.

In [ ]:
from pathlib import Path
from itertools import product
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Allows execution either from the repository root or from model_ready/.
DATA_DIR = Path('model_ready') if Path('model_ready').exists() else Path('.')
assert (DATA_DIR / 'petrol_monthly.csv').exists(), 'Run from the repository root or model_ready folder.'
CHART_DIR = DATA_DIR / 'sarimax_outputs'
CHART_DIR.mkdir(exist_ok=True)
MODEL_COLOUR = '#2a78d6'
POLICY_DATE = pd.Timestamp('2024-08-05')
VALIDATION_START = pd.Timestamp('2023-08-01')
VALIDATION_END = pd.Timestamp('2024-07-31')
TUNING_START = pd.Timestamp('2022-08-01')
TUNING_END = pd.Timestamp('2023-07-31')

In [ ]:
# Compact, deliberately varied candidate set. It covers p/q values 0, 1 and 2 and
# seasonal P/D/Q values 0 and 1 without making the daily NO2 search impractically large.
CANDIDATES = [
    ((0, 0, 0), (0, 0, 0)),
    ((1, 0, 0), (1, 0, 0)),
    ((0, 1, 1), (0, 1, 0)),
    ((1, 1, 1), (0, 1, 1)),
    ((2, 1, 1), (1, 0, 1)),
    ((1, 0, 2), (1, 1, 0)),
]

def mean_absolute_percentage_error(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    keep = np.isfinite(actual) & np.isfinite(predicted) & (actual != 0)
    return 100 * np.mean(np.abs((predicted[keep] - actual[keep]) / actual[keep]))

def percent_bias(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    keep = np.isfinite(actual) & np.isfinite(predicted)
    return 100 * (predicted[keep].sum() - actual[keep].sum()) / actual[keep].sum()

def policy_gap(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    keep = np.isfinite(actual) & np.isfinite(predicted)
    return 100 * (actual[keep].sum() - predicted[keep].sum()) / predicted[keep].sum()

def prepare_data(file_name, date_column, target, features):
    df = pd.read_csv(DATA_DIR / file_name)
    df[date_column] = pd.to_datetime(df[date_column])
    df = df.sort_values(date_column).set_index(date_column)
    # The README specifies zero for missing early rain. Zero also leaves SARIMAX with
    # a complete exogenous matrix if any other weather/control value is missing.
    X = df[features].copy().apply(pd.to_numeric, errors='coerce').fillna(0.0)
    y = pd.to_numeric(df[target], errors='coerce')
    # Keep every time point: SARIMAX supports missing endogenous observations.
    # This is essential for the README's `unused` dates and prevents time gaps.
    y = y.mask(df['split'].eq('unused'))
    # Use a positional index internally. The source timestamps do not advertise a
    # pandas frequency, and recent statsmodels versions otherwise reject forecasts.
    return df, X.astype(float).reset_index(drop=True), y.astype(float).reset_index(drop=True)

def fit_sarimax(y, X, order, seasonal_order):
    return SARIMAX(
        y, exog=X, order=order, seasonal_order=seasonal_order,
        trend='c', enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False, maxiter=200)

def tune_parameters(df, X, y, seasonal_period):
    # Only the earlier train rows are observed; Aug 2022--Jul 2023 is held back.
    tune_mask = (df.index >= TUNING_START) & (df.index <= TUNING_END) & df['split'].eq('train')
    assert tune_mask.any(), 'The required tuning holdout is not present in the train split.'
    start_pos = np.flatnonzero(tune_mask)[0]
    end_pos = np.flatnonzero(tune_mask)[-1]
    y_fit = y.iloc[:start_pos].copy()
    # The prefix can contain `unused` rows (already NaN), but cannot contain validation/policy targets.
    y_fit = y_fit.where(df['split'].iloc[:start_pos].eq('train').to_numpy())
    tuning_actual = y.iloc[start_pos:end_pos + 1]
    results = []
    for order, seasonal_short in CANDIDATES:
        seasonal_order = (*seasonal_short, seasonal_period)
        try:
            fitted = fit_sarimax(y_fit, X.iloc[:start_pos], order, seasonal_order)
            steps = int(end_pos - start_pos + 1)  # statsmodels requires a native Python int
            predicted = fitted.get_forecast(steps=steps, exog=X.iloc[start_pos:end_pos + 1]).predicted_mean
            error = mean_absolute_percentage_error(tuning_actual, predicted)
            results.append({'order': order, 'seasonal_order': seasonal_order, 'tuning_error_pct': error})
        except (ValueError, np.linalg.LinAlgError) as exc:
            results.append({'order': order, 'seasonal_order': seasonal_order, 'tuning_error_pct': np.inf, 'failure': str(exc)})
    tuning = pd.DataFrame(results).sort_values('tuning_error_pct').reset_index(drop=True)
    if not np.isfinite(tuning.loc[0, 'tuning_error_pct']):
        raise RuntimeError('All SARIMAX candidates failed; inspect the data and feature columns.')
    return tuning, tuning.loc[0, 'order'], tuning.loc[0, 'seasonal_order']

def forecast_split(df, X, y, fitted_splits, forecast_name, order, seasonal_order):
    forecast_mask = df['split'].eq(forecast_name)
    start_pos = np.flatnonzero(forecast_mask)[0]
    end_pos = np.flatnonzero(forecast_mask)[-1]
    y_fit = y.iloc[:start_pos].where(df['split'].iloc[:start_pos].isin(fitted_splits).to_numpy())
    fitted = fit_sarimax(y_fit, X.iloc[:start_pos], order, seasonal_order)
    prediction = fitted.get_forecast(
        steps=int(end_pos - start_pos + 1), exog=X.iloc[start_pos:end_pos + 1]
    ).predicted_mean
    prediction.index = df.index[start_pos:end_pos + 1]
    return prediction[forecast_mask.iloc[start_pos:end_pos + 1]]


In [ ]:
def make_charts(name, df, target, validation_pred, policy_pred, y_limits):
    predicted = pd.concat([validation_pred, policy_pred]).sort_index()
    plot = pd.DataFrame({'actual': df[target], 'predicted': predicted}).loc['2022-01-01':]
    is_no2 = name == 'no2'
    if is_no2:
        plot = plot.resample('MS').mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(plot.index, plot['actual'], color='black', label='Actual', linewidth=1.5)
    ax.plot(plot.index, plot['predicted'], color=MODEL_COLOUR, label='SARIMAX prediction', linewidth=1.5)
    ax.axvspan(VALIDATION_START, pd.Timestamp('2024-08-01'), color='lightgrey', alpha=.45, label='Validation')
    ax.axvline(POLICY_DATE, color='red', linewidth=1.3, label='50 cent fare')
    ax.set(title=f'SARIMAX: {name}', ylabel=target.replace('_', ' '), ylim=y_limits)
    ax.legend(ncol=4, fontsize=8)
    fig.tight_layout()
    fig.savefig(CHART_DIR / f'sarimax_{name}_1.png', dpi=180)
    plt.show()

    gap = pd.DataFrame({'actual': df[target], 'predicted': predicted}).loc[VALIDATION_START:]
    if is_no2:
        gap = gap.resample('MS').mean()
    gap['gap_pct'] = 100 * (gap['actual'] - gap['predicted']) / gap['predicted']
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(gap.index, gap['gap_pct'], width=20 if is_no2 else 18, color=MODEL_COLOUR)
    ax.axhline(0, color='black', linewidth=1)
    ax.axvspan(VALIDATION_START, pd.Timestamp('2024-08-01'), color='lightgrey', alpha=.45)
    ax.axvline(POLICY_DATE, color='red', linewidth=1.3)
    ax.set(title=f'SARIMAX: {name} monthly gap', ylabel='(actual − predicted) / predicted (%)',
           ylim=(-40, 40) if is_no2 else (-15, 15))
    fig.tight_layout()
    fig.savefig(CHART_DIR / f'sarimax_{name}_2.png', dpi=180)
    plt.show()

    valid = pd.DataFrame({'actual': df[target], 'predicted': validation_pred}).dropna()
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(valid['actual'], valid['predicted'], color=MODEL_COLOUR, alpha=.65)
    lo, hi = ((0, 35) if is_no2 else (45, 56))
    ax.plot([lo, hi], [lo, hi], color='black', linestyle='--', label='Predicted = actual')
    ax.set(title=f'SARIMAX: {name} validation', xlabel='Actual', ylabel='Predicted', xlim=(lo, hi), ylim=(lo, hi))
    ax.legend()
    fig.tight_layout()
    fig.savefig(CHART_DIR / f'sarimax_{name}_3.png', dpi=180)
    plt.show()

def run_model(name, file_name, date_column, target, main_features, extra_features, seasonal_period, y_limits):
    df, X_main, y = prepare_data(file_name, date_column, target, main_features)
    tuning, order, seasonal_order = tune_parameters(df, X_main, y, seasonal_period)
    print(f'\n{name.upper()} best settings: order={order}, seasonal_order={seasonal_order}')
    display(tuning)

    outcomes = []
    for run, features in [('main', main_features), ('extra', extra_features)]:
        _, X, y = prepare_data(file_name, date_column, target, features)
        validation_pred = forecast_split(df, X, y, ['train'], 'validation', order, seasonal_order)
        policy_pred = forecast_split(df, X, y, ['train', 'validation'], 'policy', order, seasonal_order)
        validation_actual = df.loc[validation_pred.index, target]
        policy_actual = df.loc[policy_pred.index, target]
        outcomes.append({
            'Model': 'SARIMAX', 'File': name, 'Run': run,
            'Error': mean_absolute_percentage_error(validation_actual, validation_pred),
            'Bias': percent_bias(validation_actual, validation_pred),
            'Gap': policy_gap(policy_actual, policy_pred),
        })
        if run == 'main':
            exported = pd.DataFrame({
                'date': pd.concat([validation_pred, policy_pred]).index,
                'actual': pd.concat([validation_actual, policy_actual]).to_numpy(),
                'predicted': pd.concat([validation_pred, policy_pred]).to_numpy(),
                'split': ['validation'] * len(validation_pred) + ['policy'] * len(policy_pred),
            })
            exported.to_csv(DATA_DIR / f'sarimax_{name}_predictions.csv', index=False)
            make_charts(name, df, target, validation_pred, policy_pred, y_limits)
    return pd.DataFrame(outcomes), tuning


In [ ]:
PETROL_FEATURES = ['ctrl_litres_pp', 'month_of_year', 'days_in_month']
PETROL_EXTRA_FEATURES = ['ctrl_all_litres_pp', 'month_of_year', 'days_in_month']
NO2_FEATURES = [
    'wind_speed', 'wind_u', 'wind_v', 'temp_c', 'temp_min_c', 'humidity_pct',
    'pressure_hpa', 'rain_mm', 'day_of_week', 'day_of_year',
    'is_public_holiday', 'is_school_holiday', 'post_covid',
]
NO2_EXTRA_FEATURES = NO2_FEATURES + ['bg_no2_ppb']

petrol_metrics, petrol_tuning = run_model(
    name='petrol', file_name='petrol_monthly.csv', date_column='month',
    target='qld_litres_pp', main_features=PETROL_FEATURES,
    extra_features=PETROL_EXTRA_FEATURES, seasonal_period=12, y_limits=(40, 60),
)
no2_metrics, no2_tuning = run_model(
    name='no2', file_name='no2_daily.csv', date_column='date',
    target='no2_ppb', main_features=NO2_FEATURES,
    extra_features=NO2_EXTRA_FEATURES, seasonal_period=7, y_limits=(0, 25),
)

summary = pd.concat([petrol_metrics, no2_metrics], ignore_index=True)
summary[['Error', 'Bias', 'Gap']] = summary[['Error', 'Bias', 'Gap']].round(2)
summary.to_csv(DATA_DIR / 'sarimax_metrics.csv', index=False)
display(summary)
print(f'Charts: {CHART_DIR.resolve()}')
print('Prediction files and SARIMAX metrics were saved in:', DATA_DIR.resolve())

## Deliverables created when the notebook runs

- `model_ready/sarimax_petrol_predictions.csv` and `model_ready/sarimax_no2_predictions.csv`: main-run validation and policy predictions in the shared layout.
- `model_ready/sarimax_metrics.csv`: validation error/bias and policy gap for both runs.
- `model_ready/sarimax_outputs/sarimax_{petrol|no2}_{1|2|3}.png`: the requested main-run charts.

A negative policy `Gap` means actual petrol use or NO₂ was lower than the SARIMAX counterfactual.